In [ ]:
# !pip install trl liger-kernel

In [ ]:
# Set GPU and environment variables
import os
GPU_DEVICES = [0]

os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'true'
os.environ["CUDA_VISIBLE_DEVICES"] = ",".join([str(i) for i in GPU_DEVICES])

In [ ]:
# Import libraries
import re
import random

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm import tqdm

from bs4 import BeautifulSoup, Comment
from datasets import load_dataset, concatenate_datasets, Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    set_seed,
)
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [ ]:
# Set all random seeds for reproducibility
def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)

    # Makes CUDA deterministic (optional)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_all_seeds(42)

In [ ]:
# Load phishing dataset (streaming) and sample N examples
N_SAMPLES = 10000

stream = load_dataset("phreshphish/phreshphish", split="train", streaming=True)
sampled = Dataset.from_list(list(stream.take(N_SAMPLES)))
print(f"Sampled {len(sampled)} examples")
print(sampled[0])

In [ ]:
# Print dataset stats and filter to English only
print(f"=== Raw sampled dataset ===")
print(f"Total samples: {len(sampled)}")
print(f"Columns: {sampled.column_names}")
lang_counts = pd.Series(sampled['lang']).value_counts()
english_count = lang_counts.get('en', 0)
print(f"English: {english_count} ({english_count/len(sampled)*100:.1f}%) | Other languages: {len(sampled) - english_count}")
print(f"Labels: {pd.Series(sampled['label']).value_counts().to_string()}")
print()

dataset = sampled.filter(lambda x: x["lang"] == "en")
non_english_count = len(sampled) - len(dataset)
print(f"=== After language filter ===")
print(f"Kept: {len(dataset)} English | Removed: {non_english_count} non-English ({non_english_count/len(sampled)*100:.1f}%)")
print(f"Labels: {pd.Series(dataset['label']).value_counts().to_string()}")

In [ ]:
# Analyze URL features for phishing vs benign
from urllib.parse import urlparse
from collections import Counter

all_urls = dataset["url"]
all_labels = dataset["label"]
phish_urls = [u for u, l in zip(all_urls, all_labels) if l == "phish"]
benign_urls = [u for u, l in zip(all_urls, all_labels) if l == "benign"]

def url_features(urls):
    tlds, url_lens, path_depths = [], [], []
    has_hyphen, has_ip, has_at = 0, 0, 0
    for url in urls:
        p = urlparse(url)
        domain = p.netloc
        parts = domain.split(".")
        if len(parts) >= 2:
            tlds.append(parts[-1])
        url_lens.append(len(url))
        path_depths.append(len([s for s in p.path.split("/") if s]))
        has_hyphen += "-" in domain
        has_ip += bool(re.match(r"\d+\.\d+\.\d+\.\d+", domain))
        has_at += "@" in url
    return tlds, url_lens, path_depths, has_hyphen, has_ip, has_at

p_tlds, p_lens, p_depths, p_hyp, p_ip, p_at = url_features(phish_urls)
b_tlds, b_lens, b_depths, b_hyp, b_ip, b_at = url_features(benign_urls)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1 - TLD comparison
top_n = 10
p_tld_counts = Counter(p_tlds)
b_tld_counts = Counter(b_tlds)
all_top_tlds = [t for t, _ in (p_tld_counts + b_tld_counts).most_common(top_n)]
x = np.arange(len(all_top_tlds))
w = 0.35
axes[0].bar(x - w/2, [100 * p_tld_counts[t] / len(phish_urls) for t in all_top_tlds], w, label="Phish", color="crimson", alpha=0.8)
axes[0].bar(x + w/2, [100 * b_tld_counts[t] / len(benign_urls) for t in all_top_tlds], w, label="Benign", color="steelblue", alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels([f".{t}" for t in all_top_tlds], rotation=45)
axes[0].set_ylabel("% of class")
axes[0].set_title("TLD Distribution")
axes[0].legend()

# 2 - URL length distribution (clip outliers at 1st/99th percentile)
all_lens = p_lens + b_lens
lo, hi = np.percentile(all_lens, 1), np.percentile(all_lens, 99)
p_lens_clip = [l for l in p_lens if lo <= l <= hi]
b_lens_clip = [l for l in b_lens if lo <= l <= hi]
shared_bins = np.linspace(lo, hi, 40)
axes[1].hist(p_lens_clip, bins=shared_bins, alpha=0.6, label="Phish", color="crimson", density=True)
axes[1].hist(b_lens_clip, bins=shared_bins, alpha=0.6, label="Benign", color="steelblue", density=True)
axes[1].set_xlabel("URL length (chars)")
axes[1].set_ylabel("Density")
axes[1].set_title("URL Length Distribution")
axes[1].legend()

# 3 - Path depth distribution
max_depth = 8
p_depth_counts = [sum(1 for d in p_depths if d == i) / len(phish_urls) * 100 for i in range(max_depth + 1)]
b_depth_counts = [sum(1 for d in b_depths if d == i) / len(benign_urls) * 100 for i in range(max_depth + 1)]
x4 = np.arange(max_depth + 1)
axes[2].bar(x4 - w/2, p_depth_counts, w, label="Phish", color="crimson", alpha=0.8)
axes[2].bar(x4 + w/2, b_depth_counts, w, label="Benign", color="steelblue", alpha=0.8)
axes[2].set_xlabel("Path depth (# of segments)")
axes[2].set_ylabel("% of class")
axes[2].set_title("URL Path Depth")
axes[2].legend()

plt.suptitle("Phishing vs Benign: URL Signal Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Phish: hyphen_in_domain={100*p_hyp/len(phish_urls):.1f}%, avg_url_len={np.mean(p_lens):.1f}, avg_path_depth={np.mean(p_depths):.1f}")
print(f"Benign: hyphen_in_domain={100*b_hyp/len(benign_urls):.1f}%, avg_url_len={np.mean(b_lens):.1f}, avg_path_depth={np.mean(b_depths):.1f}")

# --- HTML Structure Analysis (single-pass regex) ---
_re_tags = re.compile(
    r"<a\s[^>]*href\s*=|<img[\s>]|<script[\s>]|<form[\s>]"
    r"|<input\s[^>]*type\s*=\s*[\"']?password|<input[\s>]",
    re.IGNORECASE,
)
_TAG_MAP = {"a": 0, "i": 1, "s": 2, "f": 3}  # first char after '<'

all_htmls = dataset["html"]
phish_html = [h for h, l in zip(all_htmls, all_labels) if l == "phish"]
benign_html = [h for h, l in zip(all_htmls, all_labels) if l == "benign"]

def html_features(htmls):
    lengths, n_links, n_images, n_scripts, n_forms, n_inputs, n_pw = [], [], [], [], [], [], []
    for h in htmls:
        links = imgs = scripts = forms = inputs = pw = 0
        for m in _re_tags.finditer(h):
            tag_char = m.group()[1].lower()
            if tag_char == "a":
                links += 1
            elif tag_char == "i":
                s = m.group()
                if s[2].lower() == "m":
                    imgs += 1
                elif "password" in s.lower():
                    pw += 1
                    inputs += 1
                else:
                    inputs += 1
            elif tag_char == "s":
                scripts += 1
            elif tag_char == "f":
                forms += 1
        lengths.append(len(h))
        n_links.append(links)
        n_images.append(imgs)
        n_scripts.append(scripts)
        n_forms.append(forms)
        n_inputs.append(inputs)
        n_pw.append(pw)
    return lengths, n_links, n_images, n_scripts, n_forms, n_inputs, n_pw

p_len, p_links, p_imgs, p_scripts, p_forms, p_inputs, p_pw = html_features(phish_html)
b_len, b_links, b_imgs, b_scripts, b_forms, b_inputs, b_pw = html_features(benign_html)

fig2, axes2 = plt.subplots(1, 2, figsize=(12, 5))

# 1 - HTML length distribution
all_html_lens = p_len + b_len
h_lo, h_hi = np.percentile(all_html_lens, 1), np.percentile(all_html_lens, 95)
shared_html_bins = np.linspace(h_lo, h_hi, 40)
axes2[0].hist([l for l in p_len if h_lo <= l <= h_hi], bins=shared_html_bins, alpha=0.6, label="Phish", color="crimson", density=True)
axes2[0].hist([l for l in b_len if h_lo <= l <= h_hi], bins=shared_html_bins, alpha=0.6, label="Benign", color="steelblue", density=True)
axes2[0].set_xlabel("HTML length (chars)")
axes2[0].set_ylabel("Density")
axes2[0].set_title("HTML Length Distribution")
axes2[0].legend()

# 2 - Links per page distribution
all_links = p_links + b_links
l_lo, l_hi = 0, np.percentile(all_links, 95)
shared_link_bins = np.linspace(l_lo, l_hi, 40)
axes2[1].hist([l for l in p_links if l <= l_hi], bins=shared_link_bins, alpha=0.6, label="Phish", color="crimson", density=True)
axes2[1].hist([l for l in b_links if l <= l_hi], bins=shared_link_bins, alpha=0.6, label="Benign", color="steelblue", density=True)
axes2[1].set_xlabel("Number of links (<a> tags)")
axes2[1].set_ylabel("Density")
axes2[1].set_title("Links per Page")
axes2[1].legend()

plt.suptitle("Phishing vs Benign: HTML Structure Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Phish:  html_len={np.mean(p_len)/1000:.0f}K, links={np.mean(p_links):.1f}, imgs={np.mean(p_imgs):.1f}, scripts={np.mean(p_scripts):.1f}, forms={np.mean(p_forms):.1f}, inputs={np.mean(p_inputs):.1f}, pw_fields={100*sum(1 for x in p_pw if x>0)/len(p_pw):.1f}%")
print(f"Benign: html_len={np.mean(b_len)/1000:.0f}K, links={np.mean(b_links):.1f}, imgs={np.mean(b_imgs):.1f}, scripts={np.mean(b_scripts):.1f}, forms={np.mean(b_forms):.1f}, inputs={np.mean(b_inputs):.1f}, pw_fields={100*sum(1 for x in b_pw if x>0)/len(b_pw):.1f}%")

In [ ]:
# Drop unnecessary columns
drop_cols = ["sha256", "target", "date", "lang", "lang_score"]
dataset = dataset.remove_columns(drop_cols)
print(f"Removed columns: {drop_cols}")
print(f"Remaining columns: {dataset.column_names}")
print(f"Total samples: {len(dataset)}")

In [ ]:
# Define helper to plot HTML length distribution
def plot_html_length_distribution(ds):
    html_lengths = [len(x) for x in ds["html"]]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(html_lengths, bins=50, edgecolor="black", alpha=0.7)
    ax.set_xlabel("HTML length (chars)")
    ax.set_ylabel("Count")
    ax.set_title("Distribution of HTML lengths")
    ax.axvline(np.median(html_lengths), color="red", linestyle="--", label=f"Median: {np.median(html_lengths):,.0f}")
    ax.axvline(np.mean(html_lengths), color="orange", linestyle="--", label=f"Mean: {np.mean(html_lengths):,.0f}")
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(f"Min: {min(html_lengths):,} | Max: {max(html_lengths):,} | Median: {np.median(html_lengths):,.0f} | Mean: {np.mean(html_lengths):,.0f}")


plot_html_length_distribution(dataset)

In [ ]:
# Clean HTML: strip scripts/styles, keep only relevant tags and attributes
from lxml import html as lxml_html, etree

KEEP_TAGS = {"title", "meta", "form", "input", "button", "select", "textarea", "a", "img", "iframe", "label", "h1", "h2", "h3"}
KEEP_ATTRS = frozenset({"href", "src", "action", "method", "type", "name", "placeholder", "alt", "content", "value", "target"})

_REMOVE_TAGS = {"script", "style", "noscript"}

def clean_html(raw_html):
    try:
        doc = lxml_html.fromstring(raw_html)
    except Exception:
        return ""

    for el in list(doc.iter()):
        tag = el.tag
        if not isinstance(tag, str) or tag in _REMOVE_TAGS:
            p = el.getparent()
            if p is not None:
                p.remove(el)

    for el in doc.iter():
        if isinstance(el.tag, str):
            for k in [a for a in el.attrib if a not in KEEP_ATTRS]:
                try:
                    del el.attrib[k]
                except (ValueError, KeyError):
                    pass

    parts = []
    for el in doc.iter():
        if isinstance(el.tag, str) and el.tag in KEEP_TAGS:
            parts.append(etree.tostring(el, encoding="unicode", method="html", with_tail=False))
    return "\n".join(parts)

original_lengths = [len(x) for x in dataset["html"]]
dataset = dataset.map(
    lambda batch: {"html": [clean_html(h) for h in batch["html"]]},
    batched=True, batch_size=256,
)
cleaned_lengths = [len(x) for x in dataset["html"]]

print(f"=== HTML Cleaning Results ===")
print(f"Avg length before: {np.mean(original_lengths):,.0f} chars")
print(f"Avg length after:  {np.mean(cleaned_lengths):,.0f} chars")
print(f"Avg reduction:     {(1 - np.mean(cleaned_lengths)/np.mean(original_lengths))*100:.1f}%")

In [ ]:
# Plot HTML lengths after cleaning
plot_html_length_distribution(dataset)

In [ ]:
# Filter out long HTML pages and balance phish/benign classes
MAX_HTML_LEN = 20_000

before_count = len(dataset)
dataset = dataset.filter(lambda x: len(x["html"]) <= MAX_HTML_LEN)
print(f"=== HTML length filter ===")
print(f"Filtered HTML > {MAX_HTML_LEN:,} chars: {before_count} → {len(dataset)} samples (removed {before_count - len(dataset)})")
print(f"Labels: {pd.Series(dataset['label']).value_counts().to_string()}")
print()

benign = dataset.filter(lambda x: x["label"] == "benign")
phish = dataset.filter(lambda x: x["label"] == "phish")
min_count = min(len(benign), len(phish))

print(f"=== Balancing ===")
print(f"Before: benign={len(benign)}, phish={len(phish)}")
print(f"Downsampling to {min_count} per class")

benign = benign.shuffle(seed=42).select(range(min_count))
phish = phish.shuffle(seed=42).select(range(min_count))
balanced = concatenate_datasets([benign, phish]).shuffle(seed=42)

print()
print(f"=== Final balanced dataset ===")
print(f"Total: {len(balanced)} samples")
print(f"Labels: {pd.Series(balanced['label']).value_counts().to_string()}")

In [ ]:
# Split into train / val / test sets
train_test = balanced.train_test_split(test_size=0.2, seed=42)
train_ds = train_test["train"]
test_val = train_test["test"].train_test_split(test_size=0.5, seed=42)
val_ds = test_val["train"]
test_ds = test_val["test"]

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")
for name, ds in [("Train", train_ds), ("Val", val_ds), ("Test", test_ds)]:
    counts = pd.Series(ds["label"]).value_counts()
    print(f"  {name}: benign={counts.get('benign', 0)}, phish={counts.get('phish', 0)}")

In [ ]:
# Load Tokenizer and Model
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, attn_implementation="sdpa", torch_dtype=torch.bfloat16,
).to("cuda")
model = torch.compile(model)

In [ ]:
# Define prompt template and preprocessing function for chat format
prompt_template = """Analyze the following webpage and determine if it is a phishing page or a benign (legitimate) page.

URL: {url}

HTML content:
{html}

Classify this webpage as either "phish" or "benign". Output your prediction in XML format: <prediction>phish</prediction> or <prediction>benign</prediction>"""

def preprocess_function(examples):
    urls = examples["url"]
    htmls = examples["html"]
    labels = examples["label"]

    prompts = [prompt_template.format(url=url, html=html) for url, html in zip(urls, htmls)]

    messages = [
        [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": f"<prediction>{label}</prediction>"},
        ]
        for prompt, label in zip(prompts, labels)
    ]

    return {"messages": messages}


In [ ]:
# Apply preprocessing to all splits
train_ds = train_ds.map(preprocess_function, batched=True)
val_ds = val_ds.map(preprocess_function, batched=True)
test_ds = test_ds.map(preprocess_function, batched=True)

print("Sample message:")
for msg in train_ds[0]["messages"]:
    content_preview = msg["content"][:300] + "..." if len(msg["content"]) > 300 else msg["content"]
    print(f"  [{msg['role']}]: {content_preview}")

In [ ]:
# Inspect a single training sample
train_ds[0]

In [ ]:
# Define evaluation: batch inference, parse predictions, compute metrics
def evaluate_model(model, tokenizer, eval_ds, batch_size=16, max_new_tokens=128):
    model.eval()
    responses = []
    labels = []
    tokenizer.padding_side = "left"

    with torch.inference_mode():
        for start_idx in tqdm(range(0, len(eval_ds), batch_size)):
            end_idx = min(start_idx + batch_size, len(eval_ds))

            batch_messages = [eval_ds[i]["messages"][:1] for i in range(start_idx, end_idx)]
            batch_labels = [eval_ds[i]["label"] for i in range(start_idx, end_idx)]

            texts = [
                tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True,
                )
                for messages in batch_messages
            ]

            model_inputs = tokenizer(
                texts,
                return_tensors="pt",
                padding=True,
            ).to(model.device)

            generated_ids = model.generate(
                **model_inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.0,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )

            generated_ids = [
                output_ids[len(input_ids):]
                for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
            ]

            batch_responses = tokenizer.batch_decode(
                generated_ids,
                skip_special_tokens=True,
            )

            responses.extend(batch_responses)
            labels.extend(batch_labels)

    def parse_prediction(response):
        match = re.search(r"<prediction>(.*?)</prediction>", response)
        return match.group(1).strip().lower() if match else response.strip().lower()

    parsed = [parse_prediction(r) for r in responses]
    acc = accuracy_score(labels, parsed)
    valid_labels = [l for l, p in zip(labels, parsed) if p in ("phish", "benign")]
    valid_responses = [p for p in parsed if p in ("phish", "benign")]
    invalid_count = len(parsed) - len(valid_responses)

    print(f"Accuracy:  {100 * acc:.2f}%")
    if valid_responses:
        print(f"Precision: {100 * precision_score(valid_labels, valid_responses, pos_label='phish'):.2f}%")
        print(f"Recall:    {100 * recall_score(valid_labels, valid_responses, pos_label='phish'):.2f}%")
        print(f"F1:        {100 * f1_score(valid_labels, valid_responses, pos_label='phish'):.2f}%")
        print(f"\nConfusion matrix (rows=true, cols=pred):")
        cm = confusion_matrix(valid_labels, valid_responses, labels=["benign", "phish"])
        print(pd.DataFrame(cm, index=["true_benign", "true_phish"], columns=["pred_benign", "pred_phish"]))
    if invalid_count > 0:
        print(f"\nInvalid responses: {invalid_count}/{len(responses)}")
        print(f"Examples: {[r for r, p in zip(responses, parsed) if p not in ('phish', 'benign')][:5]}")

    return {
        "responses": responses,
        "parsed": parsed,
        "labels": labels,
        "accuracy": acc,
    }


In [ ]:
# Evaluate baseline (pre-training) on validation set
results = evaluate_model(model, tokenizer, val_ds)

In [ ]:
# Configure training hyperparameters and LoRA/PEFT
extra_kwargs = {
    "optim": "adamw_torch_fused",
    "learning_rate": 3e-04,
    "lr_scheduler_type": "constant_with_warmup",
    "max_grad_norm": 0.8,
    "warmup_steps": 5,
    "weight_decay": 0.1,

    "per_device_train_batch_size": 2,
    "per_device_eval_batch_size": 2,
    "gradient_accumulation_steps": 6,
    "max_steps": 100
}

peft_config = None # or peft_config = LoraConfig(???)


training_args = SFTConfig(
    output_dir="./results",    
    eval_strategy="steps",
    eval_steps=25,                  
    logging_strategy="steps",
    logging_steps=25,                   
    save_strategy="no",
    logging_dir="./logs",
    max_length=4096,
    packing=True,
    seed=42,
    data_seed=42,
    report_to="none",
    **extra_kwargs
)

In [ ]:
# Trainer Setup
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    peft_config=peft_config,
)
trainer.can_return_loss = True

In [ ]:
# Run fine-tuning
trainer.train()

In [ ]:
# Evaluate fine-tuned model on validation set
results = evaluate_model(model, tokenizer, val_ds)

In [ ]:
# Final evaluation on held-out test set
results = evaluate_model(model, tokenizer, test_ds)